# Quick start AlphaGenome in Colab

https://colab.research.google.com/

```{tip}
Open this tutorial in Google Colab for interactive viewing.
```

In [15]:
# @title Install AlphaGenome

# @markdown Run this cell to install AlphaGenome.
from IPython.display import clear_output
! pip install alphagenome
clear_output()

## Imports data & Prepares genome annotaion

In [ ]:
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# install package for loading fasta/fastq files
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.3 MB/s eta 0:00:00


In [ ]:
## load alphagenome model
dna_model = dna_client.create(colab_utils.get_api_key("Alphagenome"))

In [ ]:
## load up a GTF file containing gene and transcript locations as annotated by GENCODE (more information on GTF format
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

# Set up transcript extractors using the information in the GTF file.
# Mane select transcripts consists of of one curated transcript per locus.
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_mane_select_transcript(gtf_transcripts)
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

In [ ]:
## Given ontology that we interest
OUTPUTS  = [dna_client.OutputType.RNA_SEQ,]
ONTOLOGY   = ["UBERON:0000955"] ##brain
TARGET_LEN = dna_client.SEQUENCE_LENGTH_1MB


In [ ]:
## upload local file to cloud(colab)
from google.colab import files
uploaded = files.upload()   # 弹窗选文件
fasta_filename = list(uploaded.keys())[0]

Saving var_subset_v2.csv to var_subset_v2.csv


In [16]:
## upload local file to cloud(colab)
from google.colab import files
uploaded = files.upload()   # 弹窗选文件
fasta_filename = list(uploaded.keys())[0]

Saving output_variant.fastq to output_variant.fastq


In [ ]:
## load file in cloud into environment
## load the variants information(including chromosome, postion, ref_allele and var_allels, these information are enough for predict variants effect)
df = pd.read_csv("var_subset_v2.csv", index_col=0)
df = df.loc[0:10,]
print(f"there are {len(df)} variant")
print(df.head())

there are 10 variant
     chr        pos ref var        label  cluster                name
1   chr2  203130682   G   A  AD specific        1  chr2:203130682:G:A
2   chr2  159676533   A   G  AD specific        7  chr2:159676533:A:G
3  chr20    1914965   G   A  AD specific        4   chr20:1914965:G:A
4  chr11   86067882   C   T  AD specific        3  chr11:86067882:C:T
5   chr2  202958677   C   T  AD specific        2  chr2:202958677:C:T


In [ ]:
## similarlly load sequencing data from fastq file

from Bio import SeqIO

FASTA_FILE = "subset_variant.fastq"

sequences = []
for record in SeqIO.parse(FASTA_FILE, "fastq"):
    sequences.append({"id": record.id, "seq": str(record.seq).upper()})

print(f"\n {len(sequences)} sequences loaded!")


FileNotFoundError: [Errno 2] No such file or directory: 'subset_variant.fastq'

## Predict outputs for a DNA sequence

AlphaGenome is a model that makes predictions from DNA sequences. Let's load it
up:

In [ ]:
OUTPUTS  = [
    dna_client.OutputType.RNA_SEQ,
]
ONTOLOGY     = ["UBERON:0000955"]   # 脑
TARGET_LEN   = dna_client.SEQUENCE_LENGTH_1MB
RNA_SCORER   = variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']

In [ ]:
results    = {}   # key -> variant_output
all_scores = []

for _, row in df.iterrows():
    key = f"{row['chr']}:{row['pos']}:{row['ref']}:{row['var']}"
    print(f"\n▶ {key}")

    variant = genome.Variant(
        chromosome      = row["chr"],
        position        = int(row["pos"]),
        reference_bases = row["ref"],
        alternate_bases = row["var"],
    )
    interval = variant.reference_interval.resize(TARGET_LEN)

    # 预测
    variant_output = dna_model.predict_variant(
        interval          = interval,
        variant           = variant,
        requested_outputs = OUTPUTS,
        ontology_terms    = ONTOLOGY,
    )
    results[key] = variant_output
    print(f"  ✓ rna_seq ref shape: {variant_output.reference.rna_seq.values.shape}")

    # 打分
    variant_scores = dna_model.score_variant(
        interval        = interval,
        variant         = variant,
        variant_scorers = [RNA_SCORER],
    )
    df_scores = variant_scorers.tidy_scores(variant_scores)
    df_scores.insert(0, "variant", key)   # 加一列标记是哪个 variant
    all_scores.append(df_scores)
    print(f"  ✓ 打分完成，{len(df_scores)} 个 tracks")


▶ chr2:203130682:G:A
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，11880 个 tracks

▶ chr2:159676533:A:G
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，7524 个 tracks

▶ chr20:1914965:G:A
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，13860 个 tracks

▶ chr11:86067882:C:T
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，10692 个 tracks

▶ chr2:202958677:C:T
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，12276 个 tracks

▶ chr17:45712332:G:A
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，17820 个 tracks

▶ chr12:113151232:G:A
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，11088 个 tracks

▶ chr10:58199195:C:A
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，4752 个 tracks

▶ chr14:106647421:C:G
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，48312 个 tracks

▶ chr11:47389514:A:T
  ✓ rna_seq ref shape: (1048576, 2)
  ✓ 打分完成，17424 个 tracks


In [18]:
print(df_scores.loc[0:10])

               variant          variant_id            scored_interval  \
0   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
1   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
2   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
3   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
4   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
5   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
6   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
7   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
8   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
9   chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   
10  chr11:47389514:A:T  chr11:47389514:A>T  chr11:46865226-47913802:.   

            gene_id gene_name       gene_type gene_strand junction_Start  \
0   ENSG00000025434     NR1H3  protein_coding  

In [19]:
brain_all_scores = []

for key, df_scores in zip(results.keys(), all_scores):
    brain = df_scores[df_scores['biosample_name'].str.contains('brain', case=False)]
    brain_all_scores.append(brain)

brain_scores_df = pd.concat(brain_all_scores, ignore_index=True)
print(f"过滤后 shape: {brain_scores_df.shape}")
print(brain_scores_df[['variant', 'gene_name', 'biosample_name', 'raw_score', 'quantile_score']].head(10))

过滤后 shape: (393, 24)
              variant        gene_name biosample_name  raw_score  \
0  chr2:203130682:G:A          CYP20A1          brain  -0.000523   
1  chr2:203130682:G:A             CARF          brain  -0.001506   
2  chr2:203130682:G:A          FAM117B          brain   0.000151   
3  chr2:203130682:G:A            WDR12          brain  -0.001276   
4  chr2:203130682:G:A             ABI2          brain   0.001051   
5  chr2:203130682:G:A           NBEAL1          brain   0.000700   
6  chr2:203130682:G:A            ICA1L          brain  -0.001971   
7  chr2:203130682:G:A            RAPH1          brain  -0.000987   
8  chr2:203130682:G:A  ENSG00000202059          brain   0.000003   
9  chr2:203130682:G:A         RPL12P16          brain   0.001017   

   quantile_score  
0       -0.485882  
1       -0.743154  
2        0.312224  
3       -0.668219  
4        0.655252  
5        0.553174  
6       -0.794819  
7       -0.599367  
8        0.057616  
9        0.641886  


In [ ]:
brain_scores_df.to_csv("brain_variant_scores_v2.csv", index=False)

# 下载到本地
from google.colab import files
files.download("brain_variant_scores_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The model can make predictions for the following
[output types](https://www.alphagenomedocs.com/exploring_model_metadata.html):

AlphaGenome predicts multiple 'tracks' per output type, covering a wide variety
of tissues and cell-types. However, predictions can be made efficiently for
subsets of interest.

Here is how to make DNase-seq predictions (as specified by `OutputType`) in a
subset of tracks corresponding to lung tissue (as specified by `ontology_terms`)
for a DNA sequence of length 1Mb:

*Note: We use ontology terms from standardized biological sources like UBERON
(for anatomy) and the Cell Ontology (CL) to provide consistent and widely
recognized classifications for tissue and cell types.*

In [ ]:
## Output settings
OUTPUTS = [
    dna_client.OutputType.RNA_SEQ,
    dna_client.OutputType.DNASE,
    dna_client.OutputType.CAGE,
    dna_client.OutputType.ATAC,
]
ONTOLOGY = ["UBERON:0000955"]
TARGET_LEN = dna_client.SEQUENCE_LENGTH_16KB
print(f"目标序列长度：{TARGET_LEN:,} bp")
print(f"每条序列将在两侧填充 {(TARGET_LEN - 1000) // 2:,} 个 N")

目标序列长度：16,384 bp
每条序列将在两侧填充 7,692 个 N


In [ ]:
import numpy as np
results = {}
for entry in sequences:
    sid, seq = entry["id"], entry["seq"]
    print(f"\n▶ 预测：{sid}")
    output = dna_model.predict_sequence(
    sequence=seq.center(
        dna_client.SEQUENCE_LENGTH_1MB, 'N'
    ),  # Pad to valid sequence length.
    requested_outputs=OUTPUTS,
    ontology_terms=ONTOLOGY, ## brain
    )
    results[sid] = output

print("\n✓ 全部预测完成")



▶ 预测：chr1:23992206:G:A_ALT_1:23991706-23992706

▶ 预测：chr1:225512078:T:C_ALT_1:225511578-225512578

▶ 预测：chr2:43455496:G:A_ALT_2:43454996-43455996

▶ 预测：chr2:43505889:G:T_ALT_2:43505389-43506389

▶ 预测：chr2:43530154:A:T_ALT_2:43529654-43530654

▶ 预测：chr2:134851677:G:A_ALT_2:134851177-134852177

▶ 预测：chr2:134913732:C:T_ALT_2:134913232-134914232

▶ 预测：chr2:159628463:G:A_ALT_2:159627963-159628963

▶ 预测：chr2:159638820:C:A_ALT_2:159638320-159639320

▶ 预测：chr2:159643561:T:A_ALT_2:159643061-159644061

▶ 预测：chr2:159658883:G:A_ALT_2:159658383-159659383

▶ 预测：chr2:159667061:G:A_ALT_2:159666561-159667561

▶ 预测：chr2:159676533:A:G_ALT_2:159676033-159677033

▶ 预测：chr2:159688858:A:G_ALT_2:159688358-159689358

▶ 预测：chr2:159812177:C:G_ALT_2:159811677-159812677

▶ 预测：chr2:202913180:G:A_ALT_2:202912680-202913680

▶ 预测：chr2:202958677:C:T_ALT_2:202958177-202959177

▶ 预测：chr2:202971784:G:C_ALT_2:202971284-202972284

▶ 预测：chr2:202982094:A:T_ALT_2:202981594-202982594

▶ 预测：chr2:203104250:T:C_ALT_2:203103750-20